In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

In [2]:
df = pd.read_csv("data/raw_sensor_data.csv", header=0)

In [3]:
data = df[["Timestamp_ms","Angle_raw","Angle_proc","Velocity"]].copy()
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data = data.dropna().sort_values("Timestamp_ms").reset_index(drop=True)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Angle (rad)", "Velocity (rad/s)"),
)

fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"], y=data["Angle_raw"], name="Angle"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"], y=data["Velocity"], name="Velocity"),
    row=2, col=1
)

fig.update_xaxes(title_text="Timestamp (ms)", row=2, col=1)
fig.update_yaxes(title_text="rad", row=1, col=1)
fig.update_yaxes(title_text="rad/s", row=2, col=1)
fig.update_layout(height=600, showlegend=False)
fig.update_layout(
    title_text="Unfiltered Sensor Readings",
    showlegend=False
)
fig.show()

## Do a FFT

In [4]:
t = data["Timestamp_ms"].to_numpy() / 1000  # seconds
pend_ang = data["Angle_raw"].to_numpy()

dt = np.median(np.diff(t))
t_uniform = np.arange(t[0], t[-1], dt)
pend_ang_uniform = np.interp(t_uniform, t, pend_ang)

# Remove the mean and apply a Hann window
signal = pend_ang_uniform - np.mean(pend_ang_uniform)
window = np.hanning(len(signal))
spectrum = np.fft.rfft(signal * window)
frequencies = np.fft.rfftfreq(len(signal), d=dt)

# Normalize the one-sided amplitude spectrum
amplitude = 2 * np.abs(spectrum) / np.sum(window)
amplitude[0] /= 2
if len(signal) % 2 == 0:
    amplitude[-1] /= 2

fft_fig = go.Figure(
    go.Scatter(x=frequencies, y=amplitude, mode="lines", name="Angle FFT")
)
fft_fig.update_layout(
    title="FFT of Angle",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Amplitude (rad)",
)
fft_fig.show()

## Create a butterworth filter on raw position

In [5]:
print("Sample rate (s):", dt)
print("Sample frequency (Hz):", 1 / dt)

Sample rate (s): 0.004995000000000971
Sample frequency (Hz): 200.20020020016128


In [6]:
from scipy.signal import butter, sosfiltfilt

cutoff_hz = 7  # Adjust as needed; must be below the 250 Hz Nyquist frequency
sos = butter(4, cutoff_hz, btype="lowpass", fs=1 / dt, output="sos")
angle_filtered = sosfiltfilt(sos, pend_ang)

filtered_fig = go.Figure()
filtered_fig.add_trace(
    go.Scatter(x=(data["Timestamp_ms"].to_numpy() / 1000), y=pend_ang, name="Original", opacity=0.4)
)
filtered_fig.add_trace(
    go.Scatter(x=(data["Timestamp_ms"].to_numpy() / 1000), y=angle_filtered, name="Low-pass filtered")
)
filtered_fig.update_layout(
    title=f"Angle Low-pass Filter ({cutoff_hz} Hz cutoff)",
    xaxis_title="Time (s)",
    yaxis_title="Angle (rad)",
)
filtered_fig.show()

## Alpha-Beta Filter on Raw Position

In [7]:
ab_time = data["Timestamp_ms"].to_numpy(dtype=float) / 1000
ab_angle = data["Angle_raw"].to_numpy(dtype=float)

ab_fig = go.FigureWidget(make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Angle", "Angular rate")
))

ab_fig.add_trace(go.Scatter(
    x=ab_time, y=ab_angle, name="Raw angle",
    line=dict(color="steelblue"), opacity=0.3
), row=1, col=1)
ab_fig.add_trace(go.Scatter(
    x=ab_time, y=ab_angle, name="Filtered angle",
    line=dict(color="blue")
), row=1, col=1)

ab_fig.add_trace(go.Scatter(
    x=(data["Timestamp_ms"] / 1000), y=(data["Angle_raw"].diff())/(data["Timestamp_ms"].diff() / 1000), name="Raw delta",
    line=dict(color="indianred"), opacity=0.3
), row=2, col=1)
ab_fig.add_trace(go.Scatter(
    x=data["Timestamp_ms"] / 1000, y=data["Velocity"], name="Raw velocity",
    line=dict(color="orange"), opacity=0.3
), row=2, col=1)
ab_fig.add_trace(go.Scatter(
    x=ab_time, y=np.zeros_like(ab_time), name="Filtered rate",
    line=dict(color="red")
), row=2, col=1)

ab_fig.update_xaxes(title_text="Time (s)", row=2, col=1)
ab_fig.update_yaxes(title_text="Angle (rad)", row=1, col=1)
ab_fig.update_yaxes(title_text="Rate (rad/s)", row=2, col=1)
ab_fig.update_layout(height=650)

alpha_slider = widgets.FloatSlider(
    value=1.0, min=0.01, max=1.0, step=0.01,
    description="Alpha", continuous_update=True
)
beta_slider = widgets.FloatSlider(
    value=0.542, min=0.0, max=1.0, step=0.001,
    description="Beta", continuous_update=True, readout_format=".3f"
)

def update_alpha_beta(change=None):
    filtered_angle = np.empty_like(ab_angle)
    filtered_rate = np.empty_like(ab_angle)
    filtered_angle[0] = ab_angle[0]
    filtered_rate[0] = 0.0

    alpha, beta = alpha_slider.value, beta_slider.value
    for i in range(1, len(ab_angle)):
        elapsed = ab_time[i] - ab_time[i - 1]
        predicted = filtered_angle[i - 1] + elapsed * filtered_rate[i - 1]
        residual = ab_angle[i] - predicted
        filtered_angle[i] = predicted + alpha * residual
        filtered_rate[i] = filtered_rate[i - 1]
        if elapsed > 0:
            filtered_rate[i] += beta * residual / elapsed

    with ab_fig.batch_update():
        ab_fig.data[1].y = filtered_angle
        ab_fig.data[4].y = filtered_rate

alpha_slider.observe(update_alpha_beta, names="value")
beta_slider.observe(update_alpha_beta, names="value")
update_alpha_beta()
display(widgets.VBox([alpha_slider, beta_slider, ab_fig]))

## Alpha-Beta Filter on Low-pass Filter Data

In [8]:
ab_time = data["Timestamp_ms"].to_numpy(dtype=float) / 1000
ab_angle = angle_filtered

ab_fig = go.FigureWidget(make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Angle", "Angular rate")
))

ab_fig.add_trace(go.Scatter(
    x=ab_time, y=ab_angle, name="Raw angle",
    line=dict(color="steelblue"), opacity=0.3
), row=1, col=1)
ab_fig.add_trace(go.Scatter(
    x=ab_time, y=ab_angle, name="Filtered angle",
    line=dict(color="blue")
), row=1, col=1)
ab_fig.add_trace(go.Scatter(
    x=(data["Timestamp_ms"] / 1000), y=(data["Angle_raw"].diff())/(data["Timestamp_ms"].diff() / 1000), name="Raw rate",
    line=dict(color="indianred"), opacity=0.3
), row=2, col=1)
ab_fig.add_trace(go.Scatter(
    x=data["Timestamp_ms"] / 1000, y=data["Velocity"], name="Raw velocity",
    line=dict(color="orange"), opacity=0.3
), row=2, col=1)
ab_fig.add_trace(go.Scatter(
    x=ab_time, y=np.zeros_like(ab_time), name="Filtered rate",
    line=dict(color="red")
), row=2, col=1)

ab_fig.update_xaxes(title_text="Time (s)", row=2, col=1)
ab_fig.update_yaxes(title_text="Angle (rad)", row=1, col=1)
ab_fig.update_yaxes(title_text="Rate (rad/s)", row=2, col=1)
ab_fig.update_layout(height=650)

alpha_slider = widgets.FloatSlider(
    value=1.0, min=0.01, max=1.0, step=0.01,
    description="Alpha", continuous_update=True
)
beta_slider = widgets.FloatSlider(
    value=0.542, min=0.0, max=1.0, step=0.001,
    description="Beta", continuous_update=True, readout_format=".3f"
)

def update_alpha_beta(change=None):
    filtered_angle = np.empty_like(ab_angle)
    filtered_rate = np.empty_like(ab_angle)
    filtered_angle[0] = ab_angle[0]
    filtered_rate[0] = 0.0

    alpha, beta = alpha_slider.value, beta_slider.value
    for i in range(1, len(ab_angle)):
        elapsed = ab_time[i] - ab_time[i - 1]
        predicted = filtered_angle[i - 1] + elapsed * filtered_rate[i - 1]
        residual = ab_angle[i] - predicted
        filtered_angle[i] = predicted + alpha * residual
        filtered_rate[i] = filtered_rate[i - 1]
        if elapsed > 0:
            filtered_rate[i] += beta * residual / elapsed

    with ab_fig.batch_update():
        ab_fig.data[1].y = filtered_angle
        ab_fig.data[4].y = filtered_rate

alpha_slider.observe(update_alpha_beta, names="value")
beta_slider.observe(update_alpha_beta, names="value")
update_alpha_beta()
display(widgets.VBox([alpha_slider, beta_slider, ab_fig]))

## Time Step Analysis

In [9]:
timestamp_diffs = data["Timestamp_ms"].diff().dropna()

fig = go.Figure(go.Histogram(x=timestamp_diffs, nbinsx=50))
fig.update_layout(
    title="Distribution of Timestamp Differences",
    xaxis_title="Timestamp difference (ms)",
    yaxis_title="Count",
)
fig.show()

In [10]:
timestamp_diffs = data["Timestamp_ms"].diff()

diff_fig = go.Figure(
    go.Scatter(
        x=data.loc[timestamp_diffs.notna(), "Timestamp_ms"] / 1000,
        y=timestamp_diffs.dropna(),
        mode="lines+markers",
        name="Timestamp difference",
    )
)
diff_fig.update_layout(
    title="Timestamp Difference Over Time",
    xaxis_title="Time (s)",
    yaxis_title="Timestamp difference (ms)",
)
diff_fig.show()

In [11]:
time_s = df["Timestamp_ms"] / 1000

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_s, y=df["Angle_proc"], mode="lines", name="Angle_proc"))
fig.add_trace(go.Scatter(x=time_s, y=df["Filtered_proc"], mode="lines", name="Filtered_proc"))

fig.update_layout(
    title="Position vs. Time",
    xaxis_title="Time (s)",
    yaxis_title="Position (rad)",
)
fig.show()